In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm  # PyTorch Image Models (ResNeSt iÃƒÂ§in Ã…Å¸art)
import os
import time
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import pandas as pd
import numpy as np

# UyarÃ„Â±larÃ„Â± gizlemek iÃƒÂ§in (isteÃ„Å¸e baÃ„Å¸lÃ„Â±)
import warnings
warnings.filterwarnings("ignore")

c:\Users\Gunay\Documents\GitHub\Gastroenterology-CNN-Comparison\.venv313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- KONFÃ„Â°GÃƒÅ“RASYON ---
# ResNeSt50 Modeli (timm kÃƒÂ¼tÃƒÂ¼phanesindeki adÃ„Â±)
MODEL_NAME = 'resnest50d'

# Deney AdÃ„Â± Dosya ve klasÃƒÂ¶r isimleri buna gÃƒÂ¶re olacak
EXPERIMENT_NAME = "ResNeSt50_Baseline_MediumLarge_Run1"

# Hiperparametreler
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001
NUM_CLASSES = 8
DROPOUT_RATE = 0.4 # Overfitting'e karÃ…Å¸Ã„Â±

# DOSYA YOLLARI SUNUCUYA GÃƒâ€“RE
DATA_DIR = "../data/prepared-data"

# SonuÃƒÂ§larÃ„Â±n kaydedileceÃ„Å¸i yer
OUTPUT_DIR = f"../models/pytorch/{EXPERIMENT_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Cihaz KontrolÃƒÂ¼
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Cihaz: {DEVICE}")
print(f"Model: {MODEL_NAME}")
print(f"KayÃ„Â±t Yeri: {OUTPUT_DIR}")

Cihaz: cpu
Model: resnest50d
KayÃ„Â±t Yeri: ../models/pytorch/ResNeSt50_Baseline_MediumLarge_Run1


In [ ]:
# ImageNet Normalize DeÃ„Å¸erleri
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# Veri DÃƒÂ¶nÃƒÂ¼Ã…Å¸ÃƒÂ¼mleri
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])
}

# Datasetleri OluÃ…Å¸tur
image_datasets = {x: datasets.ImageFolder(os.path.join(DATA_DIR, x), data_transforms[x])
                  for x in ['train', 'val', 'test']}

# DataLoaderlarÃ„Â± OluÃ…Å¸tur
dataloaders = {x: DataLoader(image_datasets[x], batch_size=BATCH_SIZE,
                             shuffle=(x=='train'), num_workers=4, pin_memory=True)
               for x in ['train', 'val', 'test']}

dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val', 'test']}
class_names = image_datasets['train'].classes

print(f"SÃ„Â±nÃ„Â±flar: {class_names}")
print(f"EÃ„Å¸itim Verisi: {dataset_sizes['train']}")
print(f"Validasyon Verisi: {dataset_sizes['val']}")

In [ ]:
def create_model():
    print(f"Model indiriliyor: {MODEL_NAME}...")
    # pretrained=True ile ImageNet aÃ„Å¸Ã„Â±rlÃ„Â±klarÃ„Â±nÃ„Â± alÃ„Â±yoruz
    model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES, drop_rate=DROPOUT_RATE)
    return model

model = create_model()
model = model.to(DEVICE)

# KayÃ„Â±p Fonksiyonu ve Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Learning Rate Scheduler (Plato gÃƒÂ¶rÃƒÂ¼lÃƒÂ¼rse ÃƒÂ¶Ã„Å¸renme hÃ„Â±zÃ„Â±nÃ„Â± dÃƒÂ¼Ã…Å¸ÃƒÂ¼r)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5, verbose=True)

print("Model GPU'ya yÃƒÂ¼klendi ve eÃ„Å¸itime hazÃ„Â±r.")

In [ ]:
def train_model(model, criterion, optimizer, scheduler, num_epochs):
    since = time.time()
    best_acc = 0.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            # Batch DÃƒÂ¶ngÃƒÂ¼sÃƒÂ¼
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(DEVICE)
                labels = labels.to(DEVICE)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # GeÃƒÂ§miÃ…Å¸i Kaydet
            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())

            # En iyi modeli kaydet (Validasyon baÃ…Å¸arÃ„Â±sÃ„Â±na gÃƒÂ¶re)
            if phase == 'val':
                scheduler.step(epoch_loss)
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    save_path = os.path.join(OUTPUT_DIR, 'best_model.pth')
                    torch.save(model.state_dict(), save_path)
                    print(f"En Ã„Â°yi Model! ({best_acc:.4f}) -> Kaydedildi.")

    time_elapsed = time.time() - since
    print(f'\nEÃ„Å¸itim TamamlandÃ„Â±: {time_elapsed // 60:.0f}dk {time_elapsed % 60:.0f}sn')
    print(f'En Ã„Â°yi Validasyon DoÃ„Å¸ruluÃ„Å¸u: {best_acc:.4f}')

    # En iyi aÃ„Å¸Ã„Â±rlÃ„Â±klarÃ„Â± geri yÃƒÂ¼kle
    model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, 'best_model.pth')))
    return model, history

In [ ]:
# --- BAÃ…ÂLAT ---
model, history = train_model(model, criterion, optimizer, scheduler, num_epochs=EPOCHS)

In [ ]:
plt.figure(figsize=(14, 5))

# DoÃ„Å¸ruluk GrafiÃ„Å¸i
plt.subplot(1, 2, 1)
plt.plot(history['train_acc'], label='Train Acc')
plt.plot(history['val_acc'], label='Val Acc')
plt.title(f'{MODEL_NAME} Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# KayÃ„Â±p GrafiÃ„Å¸i
plt.subplot(1, 2, 2)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.title(f'{MODEL_NAME} Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Kaydet ve GÃƒÂ¶ster
plt.savefig(os.path.join(OUTPUT_DIR, 'training_graph.png'))
plt.show()
print(f"Grafikler kaydedildi: {os.path.join(OUTPUT_DIR, 'training_graph.png')}")

In [ ]:
print("\nTEST SETÃ„Â° DEÃ„ÂERLENDÃ„Â°RMESÃ„Â°")

model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for inputs, labels in dataloaders['test']:
        inputs = inputs.to(DEVICE)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

# 1. Classification Report (F1, Recall, Precision)
print("\nSÃ„Â±nÃ„Â±flandÃ„Â±rma Raporu:")
report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print(report)

with open(os.path.join(OUTPUT_DIR, 'classification_report.txt'), 'w') as f:
    f.write(report)

# 2. Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Tahmin Edilen')
plt.ylabel('GerÃƒÂ§ek')
plt.title(f'Confusion Matrix - {MODEL_NAME}')
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'))
plt.show()